# Colop: Parse (sentence-level)

OHCO: `parte, capit, sent_num, token_num`

Source: `../../textos/colop-pk.txt`

In [ ]:
import pandas as pd
import re

In [ ]:
src_id = 'colop'
src_path = '../../textos/colop-pk.txt'

## Parse `<PARTE>`/`<CAPITULO>` markers into chapter text

In [ ]:
lines = open(src_path, encoding='utf-8').readlines()
records, parte, capit, buf = [], None, None, []

def flush(parte, capit, buf, records):
    if buf and parte and capit:
        records.append((parte, capit, ' '.join(buf).strip()))

for raw in lines:
    line = raw.strip()
    if re.match(r'<PARTE>', line):
        flush(parte, capit, buf, records); buf = []
        parte = re.sub(r'</?PARTE>', '', line).strip(); capit = None
    elif re.match(r'<CAPITULO>', line):
        flush(parte, capit, buf, records); buf = []
        capit = re.sub(r'</?CAPITULO>', '', line).strip()
    elif line and parte and capit:
        buf.append(line)
flush(parte, capit, buf, records)

CHAP = pd.DataFrame(records, columns=['parte', 'capit', 'doc_str'])
print(f'{len(CHAP)} chapters')
CHAP.head()

## CHAP to SENT — split on sentence-terminal punctuation

In [ ]:
SENT = (
    CHAP
    .assign(doc_str=lambda df: df.doc_str.str.split(r'(?<=[.!?])\s+'))
    .explode('doc_str').dropna(subset=['doc_str'])
)
SENT = SENT[SENT.doc_str.str.strip() != ''].copy()
SENT['sent_num'] = SENT.groupby(['parte', 'capit']).cumcount()
SENT = SENT.reset_index(drop=True); SENT.index.name = 'doc_id'
DOC = SENT[['doc_str']]; DOCMAP = SENT[['parte', 'capit', 'sent_num']]
print(f'{len(DOC):,} sentences from {len(CHAP)} chapters')
DOCMAP.head()

## DOC to TOKEN

In [ ]:
TOKEN = DOC.doc_str.str.split(expand=True).stack().to_frame('token_str')
TOKEN.index.names = DOC.index.names + ['token_num']
TOKEN['term_str'] = TOKEN.token_str.str.lower().str.replace(r"[^a-z']", '', regex=True)
TOKEN = TOKEN[TOKEN.term_str != ''].dropna()
TOKEN

## Save

In [ ]:
TOKEN.to_csv(f'{src_id}-TOKEN.csv')
DOC.to_csv(f'{src_id}-DOC.csv')
DOCMAP.to_csv(f'{src_id}-DOCMAP.csv')
print('Saved to notebooks/doc_tables/')